In [ ]:
from squlearn import Executor
import squlearn
from squlearn.encoding_circuit import QiskitEncodingCircuit, HighDimEncodingCircuit, YZ_CX_EncodingCircuit, ParamZFeatureMap, HubregtsenEncodingCircuit, MultiControlEncodingCircuit, ChebyshevPQC

from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
# from qiskit import IBMQ, Aer, execute, assemble,QuantumCircuit, transpile
from qiskit import *
import matplotlib.pyplot as plt
# %matplotlib inline
import numpy as np
import math
from math import pi, sqrt
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer import Aer

from qiskit.quantum_info import Statevector, state_fidelity
# from squlearn import Executor
from FunctionEncoder import *
from scipy.special import rel_entr
from random import seed
# from random import random
from numpy import random
import cmath

import scipy as sp
import qiskit as qk
import qiskit_aer as aer
import joblib

from qiskit.circuit import QuantumCircuit, Parameter, ParameterVector
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace

In [9]:
def partial_trace_j(psi, j, b, n):
    """
    Performs the partial trace of the j-th qubit in state b.
    psi: n-qubit state vector (size 2^n)
    j: qubit index to trace out (1 to n)
    b: state of the qubit (0 or 1)
    n: total number of qubits
    Returns an (n-1)-qubit state vector.
    """
    new_dim = 2**(n-1)
    new_psi = np.zeros(new_dim, dtype=complex)
    
    # Iterate through all 2^n indices of the original state vector
    for k in range(2**n):
        # Convert k to its n-bit binary representation
        binary_rep = format(k, f'0{n}b')
        
        # Check if the j-th bit (from 0 to n-1) matches b
        # We need to adjust j because Python is 0-indexed
        if int(binary_rep[j-1]) == b:
            # Construct the new index for the (n-1)-qubit state
            # by removing the j-th bit from the binary representation
            new_binary_rep = binary_rep[:j-1] + binary_rep[j:]
            new_k = int(new_binary_rep, 2)
            
            # The coefficient is transferred to the new state
            new_psi[new_k] = psi[k]
    
    # The resulting state needs to be normalized
    return new_psi

def generalized_distance(u, v):
    """
    Calculates the generalized distance D(u, v) using the summation formula.
    D(u, v) = 1/2 * sum_{i,j} |u_i v_j - u_j v_i|^2
    
    Args:
        u (np.ndarray): A complex vector.
        v (np.ndarray): A complex vector of the same size as u.
        
    Returns:
        float: The generalized distance.
    """
    dim = len(u)
    total_sum = 0.0
    for i in range(dim):
        for j in range(dim):
            # This is the correct implementation of the term
            term = u[i] * v[j] - u[j] * v[i]
            total_sum += np.abs(term)**2
    return 0.5 * total_sum

def meyer_wallach_entanglement_measure(psi, n):
    """
    Calculates the Meyer-Wallach entanglement measure Q for an n-qubit state.
    psi: n-qubit state vector (size 2^n)
    n: total number of qubits
    """
    if len(psi) != 2**n:
        raise ValueError("The size of the state vector does not match 2^n.")

    total_D_sum = 0
    for j in range(1, n + 1):
        # Get the two (n-1)-qubit states
        psi_j_0 = partial_trace_j(psi, j, 0, n)
        psi_j_1 = partial_trace_j(psi, j, 1, n)
        
        # Calculate the generalized distance for this pair
        D_j = generalized_distance(psi_j_0, psi_j_1)
        
        # Add to the sum
        total_D_sum += D_j
        
    return (4 / n) * total_D_sum

In [ ]:
from warnings import simplefilter

n_params = 5000
def compute_expr_entg(encoder, n_params):
    num_qubits = encoder.num_qubits

    def p_harr(left_j, right_j, num_qubits):
        N = 2 ** num_qubits
        return (1 - left_j) ** (N - 1) - (1 - right_j) ** (N - 1)

    p_harr_bins = []
    for i in range(76):
        p_harr_bins.append(i/75)

    bins_x = []
    for i in range(75):
        bins_x.append(p_harr_bins[1]+p_harr_bins[i])

    p_harr_hist = []
    for i in range(75):
        p_harr_hist.append(p_harr(p_harr_bins[i], p_harr_bins[i+1], num_qubits))



    backend= Aer.get_backend('statevector_simulator')
    # backend_expr = Aer.get_backend('qasm_simulator')

    fidelity = []
    entanglement = []
    # no_similar_count = 0
    for x in range(n_params):
        data = ParameterVector("X", encoder.num_features)
        circuit_params = ParameterVector("P", encoder.num_parameters)
        # circuit_params = ParameterVector("P", 8 * 2)
        qr = QuantumRegister(num_qubits)
        cr = ClassicalRegister(num_qubits)
        qc_1 = QuantumCircuit(qr, cr)
        qc_2 = QuantumCircuit(qr, cr)

        # parameters = (2 * pi) * random.rand(len(circuit_params))
        data1 = []
        data2 = []
        for i in range(len(data)):
            data1.append(-pi + 2*pi*np.random.rand())
            data2.append(-pi + 2*pi*np.random.rand())

        theta1 = []
        theta2 = []
        for i in range (len(circuit_params)):
            theta1.append(-pi + 2*pi*np.random.rand())
            theta2.append(-pi + 2*pi*np.random.rand())
        # data_values = (-2 * pi) + ((2 * pi) - (-2 * pi)) * random.rand(5)
        # data_values = np.append(2 * (round(np.random.rand()) - 0.5), random.normal(0, 1, 4))
        # circuit = encoder.get_circuit(data_values, parameters)
        # qc = circuit2(qc, qr, theta, 1, 1)

        circuit_1 = encoder.get_circuit(data1, theta1)
        circuit_2 = encoder.get_circuit(data2, theta2)
        # circuit_1 = circuit2(qc_1, qr, theta1, 1, 0)
        # circuit_2 = circuit2(qc_2, qr, theta2, 1, 0)

        compiled_circuit1 = transpile(circuit_1, backend)
        job1 = backend.run(compiled_circuit1)
        compiled_circuit2 = transpile(circuit_2, backend)
        job2 = backend.run(compiled_circuit2)
        # job = backend.run(circuit, jobs=n_shots)
        # result = job.result()
        result1 = job1.result()
        result2 = job2.result()

        statevector_1 = result1.get_statevector()
        statevector_2 = result2.get_statevector()

        ratio = state_fidelity(statevector_1, statevector_2)
        simplefilter("ignore", category=DeprecationWarning)
        mw_metric_1 = meyer_wallach_entanglement_measure(statevector_1, num_qubits)
        # # mw_metric_2 = meyer_wallach_entanglement_measure(statevector_2, num_qubits)
        entanglement.append(mw_metric_1)

        fidelity.append(ratio)
    #Compute expressibility
    weights = np.ones_like(fidelity)/float(len(fidelity))
    P_1_hist=np.histogram(fidelity, bins=p_harr_bins, weights=weights, range=[0, 1])[0]
    expressibility = np.sum(rel_entr(P_1_hist, p_harr_hist))

    entanglement_capacity = np.sum(entanglement) / len(entanglement)
    # print(expressibility)
    return expressibility, entanglement_capacity

In [11]:
DEGREE = 0

def degree_relation_function(expression, num_parameters):
    num_params = DEGREE + 1
    if DEGREE <= 0:
        return expression
    output = 0
    # print(expression)
    params = ParameterVector("Phi", num_parameters + num_params)[num_parameters:]
    for i in range(num_params):
        output += params[i] * (expression**i)
    return output

def atan_relation_function(expression, num_parameters):
    num_params = 1
    param = ParameterVector("Phi", num_parameters + num_params)[num_parameters:]
    return param[0] * np.atan(expression)

In [ ]:
from itertools import product
from Circuit_Classes import ZZ_featuremap
import csv
num_qubits = 5
encoders = [ParamZFeatureMap, HubregtsenEncodingCircuit, HighDimEncodingCircuit, MultiControlEncodingCircuit, ChebyshevPQC, ZZ_featuremap, YZ_CX_EncodingCircuit]
funcs = [0, 1, 2, "atan"]
layers = [1, 2]

combinations = list(product(encoders, funcs, layers))

for combination in combinations:
    layer = combination[2]
    if combination[0] is ChebyshevPQC:
        # encoder = combination[0](num_qubits, num_qubits, layer, nonlinearity="arctan")
        encoder = combination[0](num_qubits, layer)
        default_circuit = encoder.get_circuit()
    elif combination[0] is ParamZFeatureMap:
        encoder = combination[0](num_qubits, num_qubits, layer, entangling=True)
    elif combination[0] is CPKernel:
        encoder = combination[0](num_qubits, layer).CPMap()
        default_circuit = encoder
    elif combination[0] is ZZ_featuremap:
        encoder = combination[0](num_qubits, layer)
        default_circuit = encoder.get_circuit()
    else:
        encoder = combination[0](num_qubits, num_qubits, layer)
    function = combination[1]
    # print(function)
    if type(function) is int:
        # print("test")
        DEGREE = function
        function = degree_relation_function
    else:
        DEGREE = "ATAN"
        function = atan_relation_function
    if not combination[0] is ZZ_featuremap:
        default_circuit = encoder.get_circuit(ParameterVector("X", encoder.num_features), ParameterVector("P", encoder.num_parameters))
        
    function_circuit = FunctionEncoder(default_circuit, function, overwrite_expression=False)
    function_encoder = QiskitEncodingCircuit(function_circuit, feature_label="X", parameter_label=["Phi", "P", "θ"])
    
    expressibility, entanglement = compute_expr_entg(function_encoder, n_params)
    print(f"For {type(encoder).__name__} encoded by {DEGREE} with {layer} layers: expr={expressibility}, entg={entanglement}")

    path = "ExprAndEntg.csv"
    output = [f"{type(encoder).__name__}-{DEGREE}-{layer}", expressibility, entanglement]

    # Open the file in append mode
    with open(path, mode='a', newline='') as file:
        writer = csv.writer(file)
        # Append the label and value as a new row
        writer.writerow(output)


For ParamZFeatureMap encoded by 0 with 1 layers: expr=0.7896125791993251, entg=0.37157481179703145
For ParamZFeatureMap encoded by 0 with 2 layers: expr=0.37788162101828615, entg=0.47334242070489346
For ParamZFeatureMap encoded by 1 with 1 layers: expr=0.6946767947560284, entg=0.39804560177036274
For ParamZFeatureMap encoded by 1 with 2 layers: expr=0.2971816148081031, entg=0.5111804814433374
For ParamZFeatureMap encoded by 2 with 1 layers: expr=0.6633215290376783, entg=0.3986319898959863
For ParamZFeatureMap encoded by 2 with 2 layers: expr=0.35114720109875075, entg=0.5110069567868257
For ParamZFeatureMap encoded by ATAN with 1 layers: expr=0.6956911773164127, entg=0.3417536211348364
For ParamZFeatureMap encoded by ATAN with 2 layers: expr=0.467314576951361, entg=0.4344458407511016
For QuantumCircuit encoded by 0 with 1 layers: expr=0.13655520044858313, entg=0.8061381492042714
For QuantumCircuit encoded by 0 with 2 layers: expr=0.04016152112746829, entg=0.8256156286712989
For QuantumC